<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Реализация класса Invoice</h2>

----

### Вариант задания №10


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Invoice в C#, который будет представлять информацию о фактурах за поставленные товары или оказанные услуги. На основе этого класса разработать 2-3 производных класса, демонстрирующих принципы наследования и полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и методы, а также переопределены некоторые методы базового класса для
демонстрации полиморфизма.
Требования к базовому классу Invoice:

• Атрибуты: Номер фактуры (InvoiceNumber), Дата выдачи (IssueDate), Общая
сумма (TotalAmount).

• Методы:

o CalculateTotal(): метод для расчета общей суммы по фактуре.

o AddLine(LineItem lineItem): метод для добавления позиции в фактуру.

o RemoveLine(LineItem lineItem): метод для удаления позиции из фактуры.

Требования к производным классам:

1. ТоварнаяФактура (GoodsInvoice): Должна содержать дополнительные атрибуты, такие как Дата поставки (SupplyDate). Метод AddLine() должен быть переопределен для добавления информации о дате поставки товара при добавлении позиции.

2. УслуговаяФактура (ServiceInvoice): Должна содержать дополнительные атрибуты, такие как Дата оказания услуги (ServiceDate). Метод RemoveLine() должен быть переопределен для добавления информации о причине аннулирования услуги при удалении позиции.

3. КомбинированнаяФактура (CombinedInvoice) (если требуется третий класс): Должна содержать дополнительные атрибуты, такие как Наличие возврата (ReturnAllowed). Метод CalculateTotal() должен быть переопределен для учета возможного возврата товара или услуги при расчете общей суммы.

#### Дополнительное задание
Добавьте к сущестующим классам конструктора классов с использованием гетторов и сетторов и реализуйте взаимодействие объектов между собой

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [4]:
using System;
using System.Collections.Generic;
using System.Linq;

// ═══════════════════════════════════════════════════════
// КЛАСС LineItem (Позиция фактуры)
// ═══════════════════════════════════════════════════════
public class LineItem
{
    private string _description;
    private decimal _quantity;
    private decimal _unitPrice;

    public string Description
    {
        get { return _description; }
        set 
        { 
            if (!string.IsNullOrWhiteSpace(value) && value.Length <= 100)
                _description = value;
            else
                throw new ArgumentException("Некорректное описание позиции");
        }
    }

    public decimal Quantity
    {
        get { return _quantity; }
        set 
        { 
            if (value > 0) _quantity = value;
            else throw new ArgumentException("Количество должно быть больше 0");
        }
    }

    public decimal UnitPrice
    {
        get { return _unitPrice; }
        set 
        { 
            if (value >= 0) _unitPrice = value;
            else throw new ArgumentException("Цена не может быть отрицательной");
        }
    }

    public decimal Total => Quantity * UnitPrice;

    public LineItem(string description, decimal quantity, decimal unitPrice)
    {
        Description = description;
        Quantity = quantity;
        UnitPrice = unitPrice;
    }

    public LineItem() : this("Без названия", 1, 0) { }

    public LineItem Clone() => new LineItem(Description, Quantity, UnitPrice);

    public virtual decimal GetTotal() => Total;

    public bool IsMoreExpensiveThan(LineItem other) => 
        other == null ? true : this.Total > other.Total;

    public bool TryMergeWith(LineItem other)
    {
        if (other != null && this.Description.Equals(other.Description, StringComparison.OrdinalIgnoreCase))
        {
            this.Quantity += other.Quantity;
            if (this.UnitPrice != other.UnitPrice)
                this.UnitPrice = (this.Total + other.Total) / this.Quantity;
            return true;
        }
        return false;
    }

    public override string ToString() => 
        $"{Description} | Кол-во: {Quantity} | Цена: {UnitPrice:C} | Сумма: {Total:C}";
}

// ═══════════════════════════════════════════════════════
// БАЗОВЫЙ КЛАСС Invoice (Фактура)
// ═══════════════════════════════════════════════════════
public class Invoice
{
    private string _invoiceNumber;
    private DateTime _issueDate;
    private decimal _totalAmount;
    protected List<LineItem> lineItems;

    public string InvoiceNumber
    {
        get { return _invoiceNumber; }
        protected set 
        { 
            if (!string.IsNullOrWhiteSpace(value) && value.Length <= 20)
                _invoiceNumber = value;
            else
                throw new ArgumentException("Некорректный номер фактуры");
        }
    }

    public DateTime IssueDate
    {
        get { return _issueDate; }
        protected set 
        { 
            if (value <= DateTime.Now) _issueDate = value;
            else throw new ArgumentException("Дата не может быть в будущем");
        }
    }

    public decimal TotalAmount
    {
        get { return _totalAmount; }
        protected set { if (value >= 0) _totalAmount = value; }
    }

    public IReadOnlyList<LineItem> LineItems => lineItems.AsReadOnly();

    public Invoice(string invoiceNumber, DateTime issueDate)
    {
        InvoiceNumber = invoiceNumber;
        IssueDate = issueDate;
        lineItems = new List<LineItem>();
        TotalAmount = 0;
    }

    public Invoice() : this("UNKNOWN", DateTime.Now) { }

    public virtual void CalculateTotal()
    {
        TotalAmount = 0;
        foreach (var item in lineItems) TotalAmount += item.GetTotal();
        Console.WriteLine($"[Invoice {InvoiceNumber}] 💰 Общая сумма: {TotalAmount:C}");
    }

    public virtual void AddLine(LineItem lineItem)
    {
        if (lineItem != null)
        {
            var existingItem = lineItems.FirstOrDefault(
                x => x.Description.Equals(lineItem.Description, StringComparison.OrdinalIgnoreCase));
            
            if (existingItem != null)
            {
                existingItem.TryMergeWith(lineItem);
                Console.WriteLine($"[Invoice {InvoiceNumber}] 📝 Позиция '{lineItem.Description}' обновлена");
            }
            else
            {
                lineItems.Add(lineItem.Clone());
                Console.WriteLine($"[Invoice {InvoiceNumber}] ➕ Добавлена: {lineItem.Description}");
            }
        }
    }

    public virtual void RemoveLine(LineItem lineItem)
    {
        if (lineItem != null)
        {
            var itemToRemove = lineItems.FirstOrDefault(
                x => x.Description.Equals(lineItem.Description, StringComparison.OrdinalIgnoreCase));
            if (itemToRemove != null)
            {
                lineItems.Remove(itemToRemove);
                Console.WriteLine($"[Invoice {InvoiceNumber}] ❌ Удалена: {lineItem.Description}");
            }
        }
    }

    public void DisplayInfo()
    {
        Console.WriteLine($"\n📄 Фактура №{InvoiceNumber}");
        Console.WriteLine($"   Дата: {IssueDate.ToShortDateString()}");
        Console.WriteLine($"   Позиций: {lineItems.Count}");
        Console.WriteLine($"   Сумма: {TotalAmount:C}");
    }

    // Взаимодействие: передача позиции
    public void TransferLineTo(Invoice targetInvoice, string description)
    {
        var item = lineItems.FirstOrDefault(
            x => x.Description.Equals(description, StringComparison.OrdinalIgnoreCase));
        if (item != null && targetInvoice != null)
        {
            lineItems.Remove(item);
            targetInvoice.AddLine(item);
            Console.WriteLine($"[🔄 TRANSFER] '{description}' из {InvoiceNumber} → {targetInvoice.InvoiceNumber}");
        }
    }

    // Взаимодействие: объединение фактур
    public void MergeWith(Invoice otherInvoice)
    {
        if (otherInvoice != null && otherInvoice != this)
        {
            Console.WriteLine($"\n[🔗 MERGE] {InvoiceNumber} + {otherInvoice.InvoiceNumber}");
            foreach (var item in otherInvoice.lineItems) this.AddLine(item);
            this.CalculateTotal();
            otherInvoice.lineItems.Clear();
            otherInvoice.TotalAmount = 0;
            Console.WriteLine($"[MERGE] {otherInvoice.InvoiceNumber} обнулена");
        }
    }

    public bool IsGreaterThan(Invoice other) => other == null ? true : this.TotalAmount > other.TotalAmount;

    public static decimal GetCombinedTotal(params Invoice[] invoices)
    {
        decimal total = 0;
        foreach (var inv in invoices) if (inv != null) total += inv.TotalAmount;
        return total;
    }
}

// ═══════════════════════════════════════════════════════
// GoodsInvoice (Товарная фактура)
// ═══════════════════════════════════════════════════════
public class GoodsInvoice : Invoice
{
    private DateTime _supplyDate;

    public DateTime SupplyDate
    {
        get { return _supplyDate; }
        private set 
        { 
            if (value >= IssueDate) _supplyDate = value;
            else throw new ArgumentException("Дата поставки не может быть раньше даты фактуры");
        }
    }

    public GoodsInvoice(string invoiceNumber, DateTime issueDate, DateTime supplyDate) 
        : base(invoiceNumber, issueDate)
    {
        SupplyDate = supplyDate;
    }

    public override void AddLine(LineItem lineItem)
    {
        base.AddLine(lineItem);
        Console.WriteLine($"[📦 GoodsInvoice] Товар '{lineItem.Description}' | Поставка: {SupplyDate.ToShortDateString()}");
    }

    public bool CanSupplyTo(GoodsInvoice other) => 
        other != null && Math.Abs((SupplyDate - other.SupplyDate).Days) <= 7;

    public GoodsInvoice CreateCombinedSupply(GoodsInvoice other, string newInvoiceNumber)
    {
        if (CanSupplyTo(other))
        {
            var combined = new GoodsInvoice(newInvoiceNumber, DateTime.Now, 
                SupplyDate > other.SupplyDate ? SupplyDate : other.SupplyDate);
            foreach (var item in this.LineItems) combined.AddLine(item.Clone());
            foreach (var item in other.LineItems) combined.AddLine(item.Clone());
            combined.CalculateTotal();
            Console.WriteLine($"[📦] Создана объединённая поставка: {newInvoiceNumber}");
            return combined;
        }
        Console.WriteLine("[📦] Невозможно объединить (даты слишком разные)");
        return null;
    }

    public void DisplaySupplyInfo() => 
        Console.WriteLine($"🚚 Дата поставки: {SupplyDate.ToShortDateString()}");
}

// ═══════════════════════════════════════════════════════
// ServiceInvoice (Услуговая фактура)
// ═══════════════════════════════════════════════════════
public class ServiceInvoice : Invoice
{
    private DateTime _serviceDate;

    public DateTime ServiceDate
    {
        get { return _serviceDate; }
        private set 
        { 
            if (value >= IssueDate.AddDays(-30)) _serviceDate = value;
            else throw new ArgumentException("Дата услуги слишком давняя");
        }
    }

    public ServiceInvoice(string invoiceNumber, DateTime issueDate, DateTime serviceDate) 
        : base(invoiceNumber, issueDate)
    {
        ServiceDate = serviceDate;
    }

    public override void RemoveLine(LineItem lineItem)
    {
        if (lineItem != null)
        {
            Console.WriteLine($"[🔧 ServiceInvoice] ⚠️ Аннулирование: '{lineItem.Description}' от {ServiceDate.ToShortDateString()}");
            base.RemoveLine(lineItem);
        }
    }

    public bool CanCancelService(ServiceInvoice other) => 
        other != null && this.LineItems.Any(i => i.Description.Contains("Услуга")) &&
                        other.LineItems.Any(i => i.Description.Contains("Услуга"));

    public void TransferServiceTo(ServiceInvoice target, string serviceName)
    {
        if (CanCancelService(target))
        {
            this.TransferLineTo(target, serviceName);
            Console.WriteLine($"[🔧] Услуга '{serviceName}' перенесена в {target.InvoiceNumber}");
        }
    }

    public void DisplayServiceInfo() => 
        Console.WriteLine($"🛠️ Дата услуги: {ServiceDate.ToShortDateString()}");
}

// ═══════════════════════════════════════════════════════
// CombinedInvoice (Комбинированная фактура)
// ═══════════════════════════════════════════════════════
public class CombinedInvoice : Invoice
{
    private bool _returnAllowed;
    private decimal _returnAmount;

    public bool ReturnAllowed
    {
        get { return _returnAllowed; }
        private set { _returnAllowed = value; }
    }

    public decimal ReturnAmount
    {
        get { return _returnAmount; }
        private set { if (value >= 0 && value <= TotalAmount) _returnAmount = value; }
    }

    public CombinedInvoice(string invoiceNumber, DateTime issueDate, bool returnAllowed) 
        : base(invoiceNumber, issueDate)
    {
        ReturnAllowed = returnAllowed;
        ReturnAmount = 0;
    }

    public override void CalculateTotal()
    {
        base.CalculateTotal();
        if (ReturnAllowed && ReturnAmount > 0)
        {
            decimal finalAmount = TotalAmount - ReturnAmount;
            Console.WriteLine($"[🔄 CombinedInvoice] ↩️ Возврат: {ReturnAmount:C}");
            Console.WriteLine($"[🔄] ✅ Итог: {finalAmount:C}");
            TotalAmount = finalAmount;
        }
        else if (ReturnAllowed)
        {
            Console.WriteLine($"[🔄] Возврат разрешён, но возвратов не было");
        }
    }

    public void AddReturn(decimal amount)
    {
        if (ReturnAllowed && amount >= 0)
        {
            ReturnAmount += amount;
            Console.WriteLine($"[🔄] ➕ Возврат: {amount:C}");
            CalculateTotal();
        }
        else if (!ReturnAllowed)
        {
            Console.WriteLine("[🔄] ❌ Возврат не разрешён!");
        }
    }

    public void ApplyReturnFrom(CombinedInvoice other)
    {
        if (ReturnAllowed && other != null && other.ReturnAmount > 0)
        {
            Console.WriteLine($"[🔄] Применяем возврат из {other.InvoiceNumber}");
            this.AddReturn(other.ReturnAmount);
        }
    }

    public void ImportFromGoodsInvoice(GoodsInvoice goodsInv)
    {
        if (goodsInv != null)
        {
            Console.WriteLine($"[🔄] 📦 Импорт товаров из {goodsInv.InvoiceNumber}");
            foreach (var item in goodsInv.LineItems) this.AddLine(item.Clone());
            this.CalculateTotal();
        }
    }

    public void ImportFromServiceInvoice(ServiceInvoice serviceInv)
    {
        if (serviceInv != null)
        {
            Console.WriteLine($"[🔄] 🔧 Импорт услуг из {serviceInv.InvoiceNumber}");
            foreach (var item in serviceInv.LineItems) this.AddLine(item.Clone());
            this.CalculateTotal();
        }
    }

    public void DisplayReturnInfo()
    {
        Console.WriteLine($"↩️ Возврат: {(ReturnAllowed ? "✅ Да" : "❌ Нет")}");
        if (ReturnAmount > 0) Console.WriteLine($"   Сумма возврата: {ReturnAmount:C}");
    }
}

// ═══════════════════════════════════════════════════════
// 🎬 ДЕМОСТРАЦИЯ РАБОТЫ КЛАССОВ
// ═══════════════════════════════════════════════════════

Console.WriteLine("╔════════════════════════════════════════╗");
Console.WriteLine("║  📄 ДЕМОСТРАЦИЯ: ФАКТУРЫ И ПОЗИЦИИ 📄  ║");
Console.WriteLine("╚════════════════════════════════════════╝\n");

// ─────────────────────────────────────────────────────────
// 1️⃣ СОЗДАНИЕ ОБЪЕКТОВ (конструкторы + валидация)
// ─────────────────────────────────────────────────────────
Console.WriteLine("📦 1. СОЗДАНИЕ ФАКТУР:");
Console.WriteLine("──────────────────────");
var goodsInv = new GoodsInvoice("GI-001", new DateTime(2024, 1, 15), new DateTime(2024, 1, 20));
var serviceInv = new ServiceInvoice("SI-001", new DateTime(2024, 1, 16), new DateTime(2024, 1, 25));
var combinedInv = new CombinedInvoice("CI-001", new DateTime(2024, 1, 17), true);
Console.WriteLine("✓ Создано: товарная, услуговая, комбинированная фактуры ✓\n");

// ─────────────────────────────────────────────────────────
// 2️⃣ ДОБАВЛЕНИЕ ПОЗИЦИЙ (сеттеры + валидация)
// ─────────────────────────────────────────────────────────
Console.WriteLine("📝 2. ДОБАВЛЕНИЕ ПОЗИЦИЙ:");
Console.WriteLine("─────────────────────────");
goodsInv.AddLine(new LineItem("Ноутбук", 2, 1500));
goodsInv.AddLine(new LineItem("Мышь", 5, 25));
goodsInv.AddLine(new LineItem("Мышь", 3, 30)); // Та же позиция → объединение!

serviceInv.AddLine(new LineItem("Консультация", 10, 100));

combinedInv.AddLine(new LineItem("Товар 1", 3, 200));
combinedInv.AddLine(new LineItem("Услуга 1", 1, 500));
Console.WriteLine();

// ─────────────────────────────────────────────────────────
// 3️⃣ ВЗАИМОДЕЙСТВИЕ: ПЕРЕДАЧА ПОЗИЦИИ
// ─────────────────────────────────────────────────────────
Console.WriteLine("🔄 3. ПЕРЕДАЧА ПОЗИЦИИ МЕЖДУ ФАКТУРАМИ:");
Console.WriteLine("───────────────────────────────────────");
goodsInv.TransferLineTo(combinedInv, "Мышь");
Console.WriteLine();

// ─────────────────────────────────────────────────────────
// 4️⃣ ВЗАИМОДЕЙСТВИЕ: ОБЪЕДИНЕНИЕ ФАКТУР
// ─────────────────────────────────────────────────────────
Console.WriteLine("🔗 4. ОБЪЕДИНЕНИЕ ФАКТУР:");
Console.WriteLine("─────────────────────────");
serviceInv.MergeWith(combinedInv);
Console.WriteLine();

// ─────────────────────────────────────────────────────────
// 5️⃣ ВЗАИМОДЕЙСТВИЕ: СРАВНЕНИЕ
// ─────────────────────────────────────────────────────────
Console.WriteLine("⚖️ 5. СРАВНЕНИЕ ФАКТУР:");
Console.WriteLine("───────────────────────");
Console.WriteLine($"{goodsInv.InvoiceNumber} > {combinedInv.InvoiceNumber}? {goodsInv.IsGreaterThan(combinedInv)}");
Console.WriteLine();

// ─────────────────────────────────────────────────────────
// 6️⃣ ВЗАИМОДЕЙСТВИЕ: ОБЩАЯ СУММА
// ─────────────────────────────────────────────────────────
Console.WriteLine("💰 6. ОБЩАЯ СУММА НЕСКОЛЬКИХ ФАКТУР:");
Console.WriteLine("────────────────────────────────────");
decimal grandTotal = Invoice.GetCombinedTotal(goodsInv, serviceInv, combinedInv);
Console.WriteLine($"📊 Сумма всех фактур: {grandTotal:C}\n");

// ─────────────────────────────────────────────────────────
// 7️⃣ ВЗАИМОДЕЙСТВИЕ: ОБЪЕДИНЕНИЕ ПОСТАВОК
// ─────────────────────────────────────────────────────────
Console.WriteLine("📦 7. ОБЪЕДИНЕНИЕ ТОВАРНЫХ ПОСТАВОК:");
Console.WriteLine("────────────────────────────────────");
var goodsInv2 = new GoodsInvoice("GI-002", new DateTime(2024, 1, 16), new DateTime(2024, 1, 22));
goodsInv2.AddLine(new LineItem("Клавиатура", 10, 50));

var combinedSupply = goodsInv.CreateCombinedSupply(goodsInv2, "GI-COMBINED");
if (combinedSupply != null) combinedSupply.DisplayInfo();
Console.WriteLine();

// ─────────────────────────────────────────────────────────
// 8️⃣ ВЗАИМОДЕЙСТВИЕ: ВОЗВРАТЫ
// ─────────────────────────────────────────────────────────
Console.WriteLine("↩️ 8. ОБРАБОТКА ВОЗВРАТОВ:");
Console.WriteLine("──────────────────────────");
var combinedInv2 = new CombinedInvoice("CI-002", new DateTime(2024, 1, 18), true);
combinedInv2.AddLine(new LineItem("Возвратный товар", 1, 300));
combinedInv2.AddReturn(300);

combinedInv.ApplyReturnFrom(combinedInv2);
Console.WriteLine();

// ─────────────────────────────────────────────────────────
// 9️⃣ ВЗАИМОДЕЙСТВИЕ: ИМПОРТ В КОМБИНИРОВАННУЮ
// ─────────────────────────────────────────────────────────
Console.WriteLine("🔄 9. ИМПОРТ ИЗ РАЗНЫХ ТИПОВ ФАКТУР:");
Console.WriteLine("────────────────────────────────────");
var newCombined = new CombinedInvoice("CI-NEW", DateTime.Now, true);
newCombined.ImportFromGoodsInvoice(goodsInv);
newCombined.ImportFromServiceInvoice(serviceInv);
newCombined.DisplayInfo();
Console.WriteLine();

// ─────────────────────────────────────────────────────────
// 🔟 ВАЛИДАЦИЯ ЧЕРЕЗ СЕТТЕРЫ
// ─────────────────────────────────────────────────────────
Console.WriteLine("🛡️ 10. ПРОВЕРКА ВАЛИДАЦИИ (сеттеры):");
Console.WriteLine("────────────────────────────────────");
try
{
    var invalidItem = new LineItem("", -5, -100);
}
catch (ArgumentException ex)
{
    Console.WriteLine($"❌ Ошибка валидации: {ex.Message}");
}

try
{
    var futureDate = new Invoice("TEST", DateTime.Now.AddDays(10));
}
catch (ArgumentException ex)
{
    Console.WriteLine($"❌ Ошибка даты: {ex.Message}");
}
Console.WriteLine();

// ─────────────────────────────────────────────────────────
// 📋 ФИНАЛЬНЫЙ ОТЧЁТ
// ─────────────────────────────────────────────────────────
Console.WriteLine("╔════════════════════════════════════════╗");
Console.WriteLine("║           📋 ФИНАЛЬНЫЙ ОТЧЁТ           ║");
Console.WriteLine("╚════════════════════════════════════════╝");

Console.WriteLine("\n📄 Товарная фактура:");
goodsInv.DisplayInfo();
goodsInv.DisplaySupplyInfo();

Console.WriteLine("\n🔄 Комбинированная фактура:");
combinedInv.DisplayInfo();
combinedInv.DisplayReturnInfo();

Console.WriteLine($"\n✅ Всего создано фактур: 6");
Console.WriteLine("✅ Всего создано позиций: 8+");

Console.WriteLine("\n🎯 РЕАЛИЗОВАННЫЕ ПРИНЦИПЫ ООП:");
Console.WriteLine("  ✓ Инкапсуляция: приватные поля + свойства с валидацией");
Console.WriteLine("  ✓ Наследование: GoodsInvoice, ServiceInvoice, CombinedInvoice → Invoice");
Console.WriteLine("  ✓ Полиморфизм: переопределение AddLine(), RemoveLine(), CalculateTotal()");
Console.WriteLine("  ✓ Конструкторы: с параметрами + по умолчанию + цепочка вызовов");
Console.WriteLine("  ✓ Взаимодействие объектов: TransferLineTo, MergeWith, ImportFrom...");
Console.WriteLine("  ✓ Статические члены: GetCombinedTotal()");

Console.WriteLine("\n🎉 Демонстрация завершена успешно! 🎉");

╔════════════════════════════════════════╗
║  📄 ДЕМОСТРАЦИЯ: ФАКТУРЫ И ПОЗИЦИИ 📄  ║
╚════════════════════════════════════════╝

📦 1. СОЗДАНИЕ ФАКТУР:
──────────────────────
✓ Создано: товарная, услуговая, комбинированная фактуры ✓

📝 2. ДОБАВЛЕНИЕ ПОЗИЦИЙ:
─────────────────────────
[Invoice GI-001] ➕ Добавлена: Ноутбук
[📦 GoodsInvoice] Товар 'Ноутбук' | Поставка: 20.01.2024
[Invoice GI-001] ➕ Добавлена: Мышь
[📦 GoodsInvoice] Товар 'Мышь' | Поставка: 20.01.2024
[Invoice GI-001] 📝 Позиция 'Мышь' обновлена
[📦 GoodsInvoice] Товар 'Мышь' | Поставка: 20.01.2024
[Invoice SI-001] ➕ Добавлена: Консультация
[Invoice CI-001] ➕ Добавлена: Товар 1
[Invoice CI-001] ➕ Добавлена: Услуга 1

🔄 3. ПЕРЕДАЧА ПОЗИЦИИ МЕЖДУ ФАКТУРАМИ:
───────────────────────────────────────
[Invoice CI-001] ➕ Добавлена: Мышь
[🔄 TRANSFER] 'Мышь' из GI-001 → CI-001

🔗 4. ОБЪЕДИНЕНИЕ ФАКТУР:
─────────────────────────

[🔗 MERGE] SI-001 + CI-001
[Invoice SI-001] ➕ Добавлена: Товар 1
[Invoice SI-001] ➕ Добавлена: Услуга 1
[Invoice

Error: System.ArgumentException: Дата поставки не может быть раньше даты фактуры
   at Submission#4.GoodsInvoice.set_SupplyDate(DateTime value)
   at Submission#4.GoodsInvoice..ctor(String invoiceNumber, DateTime issueDate, DateTime supplyDate)
   at Submission#4.GoodsInvoice.CreateCombinedSupply(GoodsInvoice other, String newInvoiceNumber)
   at Submission#4.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)